## 2장 1강 : System·User·Assistant 메시지 구조

### 3. 역할별 메시지 템플릿 조립하기

- System: 모델의 역할(페르소나), 상황, 과업, 규칙 정의, 최우선순위 / `SystemMessage`
- User: 실제 질의 응답에 대한 사용자 질문, 처리할 데이터의 전달, 가변적, 프롬프트 템플릿 / `HumanMessage`
- Assistant: AI가 출력한 응답 / `AIMessage`

- User - Assistant => 대화의 맥락 정보를 쌍으로 전달, 1쌍 - 턴, 여러쌍 -> 멀티 턴
- BaseMessage의 하위 클래스 (HumanMessage, SystemMessage, AIMessage, ToolMessage)

#### 3.1 라이브러리 로드 및 환경 설정

In [19]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage

model = init_chat_model(
    model_provider="ollama",
    model="mistral"
)

parser = StrOutputParser()

chain = model | parser

#### 3.2 역할별 메시지 템플릿 생성 및 실행

In [20]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 재치있게 컴퓨터 개념을 설명하는 유치원 교사입니다"), 
    ("user", "다음 개념을 설명하세요: {text}")
])

chain2 = prompt_template | model | parser

response = chain2.invoke({"text": "함수"})
response

' 함수는 특정 작업을 수행하는 코드 조각으로, 컴퓨터 프로그램에서 반복적인 작업을 간소화하고 재사용 가능하게 하는 것이 목적입니다. 함수는 이름, 입력(매개변수), 작업, 출력(반환 값)을 가지며 일반적으로 다음과 같은 형식으로 정의됩니다.\n\n```python\n함수 이름(매개변수1, 매개변수2, ...) {\n  코드 조각\n}\n```\n\n예를 들어, 두 수를 입력받아 덧셈 결과를 반환하는 함수는 다음과 같이 정의할 수 있습니다.\n\n```python\ndef add(a, b):\n  결과 = a + b\n  return 결과\n```\n\n이렇게 정의된 함수는 아래와 같이 사용할 수 있습니다.\n\n```python\n결과 = add(3, 5)\nprint(결과) # 결과는 8이 됩니다.\n```\n\n함수는 코드를 구조화하고 유지 보수를 용이하게 하는 중요한 요소입니다. 특히, 큰 프로그램에서는 같은 작업을 반복적으로 수행하는 것을 피하고 싶고, 코드 구조를 명확하게 하기 위해서는 함수를 사용하는 것이 중요합니다.'

### 4. 대화 이력(Memory) 구조 설계하기

#### 4.1 Assistant 메시지 기반 문맥 유지

In [23]:
messages: list[BaseMessage] = [
    HumanMessage("내 이름은 김철수야. 만나서 반가워."),
    AIMessage("반갑습니다 철수님. 무엇을 도와드릴까요?"),
    HumanMessage("오늘 점심 메뉴를 추천해줘"),
    AIMessage("오늘 점심은 돈가스가 좋을 것 같아요."),
    HumanMessage("오늘 점심 추천이 뭐였지?")
    #HumanMessage("내 이름이 뭐라고 했지?")
]

response = chain.invoke(messages)
response

' 오늘 점심에는 돈가스가 추천이 되었어요.'